# 11 · Do modelo à demonstração: Gradio

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Um modelo dentro de um notebook só é útil para quem sabe executar notebooks. Para que um gestor, um cliente ou um colega de outra área possa experimentar, questionar e dar retorno, o modelo precisa de uma **interface**. Esta aula fecha o percurso do curso: pega um modelo treinado e o transforma em uma aplicação web, com poucas dezenas de linhas de Python.

## O que você será capaz de fazer ao final

1. Separar o **artefato** do modelo (um arquivo) do código que o treina e do código que o serve.
2. Escrever e **testar** a função de previsão antes de qualquer interface.
3. Montar uma interface com `gr.Interface` e uma aplicação completa com `gr.Blocks`.
4. Mostrar ao usuário **por que** o modelo respondeu o que respondeu.
5. Coletar retorno dos usuários e processar arquivos em lote.
6. Transformar o notebook em um `app.py` pronto para publicar, com as dependências fixadas.
7. Decidir o que uma demonstração deve avisar e o que ela não deve guardar.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:          # no Colab, o Gradio precisa ser instalado
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)

import csv
import string
import warnings
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import sklearn

SEMENTE = 42
print("gradio", gr.__version__, "· scikit-learn", sklearn.__version__)

## 1. Por que uma demonstração

| Sem interface | Com interface |
|---|---|
| "O modelo tem AUC de 0,96." | "Digite uma avaliação e veja o que ele responde." |
| só a equipe técnica experimenta | quem conhece o negócio encontra os erros que a métrica esconde |
| o retorno chega em reuniões | o retorno chega enquanto a pessoa usa |

Três ferramentas cobrem quase todos os casos, e as três usam só Python:

| Ferramenta | Melhor para | Como se escreve |
|---|---|---|
| **Gradio** | demonstrar um modelo: entrada, saída, exemplos | uma função e a lista de componentes |
| **Streamlit** | painéis e aplicações de dados com várias telas | um roteiro executado de cima para baixo |
| **FastAPI** | servir o modelo para outros sistemas | rotas HTTP; sem tela |

Usamos o Gradio porque é o caminho mais curto entre uma função e uma tela, roda dentro do notebook e é o padrão dos *Spaces* do Hugging Face. As outras duas aparecem nos slides de entrega.

## 2. O modelo como artefato

Uma aplicação **não treina** o modelo: ela carrega um arquivo pronto. Esse arquivo, o **artefato**, é a fronteira entre o trabalho de modelagem e o de entrega.

Para que este notebook funcione sozinho, a célula abaixo refaz, em poucos segundos, o modelo de texto da aula 05 (TF-IDF e regressão logística) e o grava em disco. Em um projeto, essa parte ficaria em outro arquivo, e a aplicação só conheceria o `.joblib`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend"])
textos = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
alvo = (df["recommend_to_a_friend"] == "No").astype(int)
t_treino, t_teste, y_treino, y_teste = train_test_split(textos, alvo, test_size=0.2, stratify=alvo, random_state=SEMENTE)

treinado = Pipeline([("tfidf", TfidfVectorizer(min_df=5)), ("clf", LogisticRegression(max_iter=1000))]).fit(t_treino, y_treino)
auc_teste = roc_auc_score(y_teste, treinado.predict_proba(t_teste)[:, 1])

joblib.dump(treinado, "modelo_avaliacoes.joblib")
print(f"modelo_avaliacoes.joblib: {Path('modelo_avaliacoes.joblib').stat().st_size / 1e6:.1f} MB · AUC no teste: {auc_teste:.3f}")

Daqui em diante, o notebook se comporta como a aplicação: esquece o treino e **carrega o arquivo**.

In [ ]:
modelo = joblib.load("modelo_avaliacoes.joblib")
print(modelo)

> ⚠️ **Um arquivo `.joblib` só é confiável com a mesma versão do scikit-learn que o gravou.** Carregá-lo em outra versão pode falhar ou, pior, funcionar com resultados diferentes. Por isso as versões das bibliotecas são parte do artefato (seção 9). E como o formato executa código ao ser carregado, nunca abra um `.joblib` de origem desconhecida.

## 3. A primeira interface

O Gradio precisa de três coisas: uma **função**, a descrição das **entradas** e a das **saídas**.

A função abaixo recebe um texto e devolve um dicionário com a probabilidade de cada classe, que é o formato esperado pelo componente `gr.Label`.

In [ ]:
def prever(texto):
    """Recebe o texto de uma avaliação e devolve as probabilidades das duas classes."""
    p = float(modelo.predict_proba([texto])[0, 1])
    return {"não recomenda": p, "recomenda": 1 - p}


prever("Produto chegou quebrado e ninguém responde. Péssimo.")

In [ ]:
demo = gr.Interface(
    fn=prever,
    inputs=gr.Textbox(label="Avaliação do cliente", lines=4, placeholder="Cole ou digite uma avaliação"),
    outputs=gr.Label(label="O cliente recomenda o produto?"),
    examples=["Chegou antes do prazo, produto excelente!", "Não recebi o produto e não consigo o reembolso.", "É bom, mas esperava mais pelo preço."],
    title="Avaliações de clientes",
    description="Modelo treinado com avaliações de uma loja virtual brasileira (B2W-Reviews01).",
    flagging_mode="never",
)
demo.launch(quiet=True)

Ao executar a célula, a aplicação aparece dentro do notebook. Experimente os exemplos e escreva suas próprias avaliações.

| Você deve saber | |
|---|---|
| **Onde o modelo roda** | no computador (ou na sessão do Colab) que executou `launch()`. Fechar o notebook derruba a aplicação. |
| **Link público** | no Colab, o Gradio cria sozinho um endereço temporário `https://....gradio.live`, que funciona por até uma semana enquanto a sessão estiver viva. Fora do Colab, peça com `launch(share=True)`. |
| **Quem tem o link usa a sua máquina** | não compartilhe um link de demonstração que dê acesso a dados sensíveis; para restringir, use `launch(auth=("usuario", "senha"))`. |
| **Erros** | `launch(debug=True)` mostra no notebook os erros da função, e mantém a célula ocupada enquanto a aplicação estiver aberta. |

Para encerrar uma aplicação:

In [ ]:
demo.close()

### ✏️ Exercício 1 (★): a sua primeira interface

Escreva a função `resumir(texto)`, que devolve uma frase como `"Probabilidade de não recomendar: 87%"` (percentual sem casas decimais), e monte `demo_ex1`, uma `gr.Interface` com uma caixa de texto de entrada e uma caixa de texto de saída.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
def resumir(texto):
    ...


demo_ex1 = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
_frase = resumir("Não recebi o produto. Péssimo atendimento, não recomendo.")
assert isinstance(_frase, str) and _frase.startswith("Probabilidade de não recomendar: ") and _frase.endswith("%"), "confira o formato da frase"
assert "." not in _frase.split(": ")[1] and "," not in _frase.split(": ")[1], "o percentual não deve ter casas decimais"
assert int(_frase.split(": ")[1].rstrip("%")) > 50 and int(resumir("Excelente, recomendo!").split(": ")[1].rstrip("%")) < 50
assert isinstance(demo_ex1, gr.Interface), "demo_ex1 deve ser uma gr.Interface"
print("Exercício 1 correto:", _frase, "· para ver a interface, execute demo_ex1.launch()")

## 4. Testar antes de mostrar

A função de previsão é Python comum. Dá para testá-la sem abrir tela nenhuma, e é assim que se deve fazer: quando alguém estiver usando a demonstração, é tarde para descobrir que um texto vazio derruba a aplicação.

🔮 **Preveja:** o que a função `prever` devolve para um texto vazio? E para um texto em outro idioma?

In [ ]:
for caso in ["", "   ", "asdf qwer zxcv", "The product is great, I love it!", "ótimo " * 3000]:
    resultado = prever(caso)
    print(f"{caso[:35]!r:<40} → não recomenda: {resultado['não recomenda']:.3f}")

print(f"\nProporção de 'não recomenda' no treino: {y_treino.mean():.3f}")

Nada quebra, e esse é o problema.

- Para o texto vazio e para letras ao acaso, nenhuma palavra do vocabulário foi encontrada, e a "previsão" (0,34) é só o termo constante do modelo. Ela não depende do texto, e nem sequer coincide com a proporção de "não recomenda" no treino.
- O texto em inglês é um caso pior: algumas de suas palavras coincidem com palavras do vocabulário, e o modelo devolve outro número, como se tivesse entendido.
- A palavra "ótimo" repetida três mil vezes leva a probabilidade a zero, com toda a confiança.

**Um modelo sempre responde.** Cabe à aplicação decidir quando a resposta não merece ser mostrada. Vamos escrever a função que a interface vai usar de verdade, com três cuidados:

- recusar entradas vazias, com uma mensagem clara (`gr.Error` vira um aviso na tela);
- avisar quando nenhuma palavra do texto é conhecida pelo modelo;
- transformar a probabilidade em uma **decisão**, usando um limiar que o usuário pode ajustar (aula 05: o limiar depende dos custos de cada tipo de erro).

In [ ]:
vetorizador = modelo.named_steps["tfidf"]
classificador = modelo.named_steps["clf"]
MAX_CARACTERES = 5000


def analisar(texto, limiar=0.5):
    """Função principal da aplicação: valida a entrada, prevê e decide."""
    texto = (texto or "").strip()
    if not texto:
        raise gr.Error("Digite uma avaliação antes de analisar.")
    texto = texto[:MAX_CARACTERES]

    palavras_conhecidas = vetorizador.transform([texto]).nnz              # quantas palavras do vocabulário aparecem
    if palavras_conhecidas == 0:
        return {}, "Não foi possível analisar: nenhuma palavra do texto é conhecida pelo modelo (o texto está em português?)."

    p = float(modelo.predict_proba([texto])[0, 1])
    decisao = "Encaminhar para o atendimento" if p >= limiar else "Nenhuma ação necessária"
    return {"não recomenda": p, "recomenda": 1 - p}, f"{decisao} (probabilidade {p:.0%}, limiar {limiar:.0%})"

In [ ]:
# Testes da função, sem interface
probabilidades, decisao = analisar("Produto chegou quebrado e ninguém responde.", limiar=0.5)
assert abs(sum(probabilidades.values()) - 1) < 1e-9 and decisao.startswith("Encaminhar")

assert analisar("Produto excelente, entrega rápida, recomendo!", 0.5)[1].startswith("Nenhuma ação")
assert analisar("asdf qwer zxcv")[0] == {} and "Não foi possível" in analisar("asdf qwer zxcv")[1]
p_morna = analisar("É bom, mas esperava mais.")[0]["não recomenda"]                    # o limiar muda a decisão, e não a probabilidade
assert analisar("É bom, mas esperava mais.", limiar=p_morna + 0.01)[1].startswith("Nenhuma ação")
assert analisar("É bom, mas esperava mais.", limiar=p_morna - 0.01)[1].startswith("Encaminhar")

for entrada_invalida in ["", "   ", None]:
    try:
        analisar(entrada_invalida)
        raise AssertionError("deveria ter recusado a entrada vazia")
    except gr.Error as erro:
        mensagem = str(erro)                # a variável `erro` deixa de existir ao sair do except
print("Todos os testes passaram. Mensagem para entrada vazia:", mensagem)

## 5. Explicar a previsão

Uma probabilidade sem explicação convida à desconfiança, ou à confiança cega. Nosso modelo é linear, e isso permite uma explicação exata: a contribuição de cada palavra é o **peso da palavra no texto (TF-IDF) multiplicado pelo coeficiente** que o modelo aprendeu para ela. Somando todas as contribuições e o termo constante, chega-se às log-chances da previsão.

In [ ]:
def contribuicoes(texto):
    """Contribuição de cada palavra conhecida do texto para as log-chances de 'não recomenda'."""
    linha = vetorizador.transform([texto])
    valores = linha.multiply(classificador.coef_).toarray()[0]
    presentes = linha.nonzero()[1]
    return pd.Series(valores[presentes], index=vetorizador.get_feature_names_out()[presentes]).sort_values()


exemplo = "O produto é bonito, mas chegou com defeito e o atendimento não resolveu."
c = contribuicoes(exemplo)
print(c.round(2).to_string())

log_chances = classificador.intercept_[0] + c.sum()
print(f"\nconstante {classificador.intercept_[0]:.2f} + contribuições {c.sum():.2f} = {log_chances:.2f} → probabilidade {1 / (1 + np.exp(-log_chances)):.3f}")
print(f"conferindo com o modelo:                              probabilidade {modelo.predict_proba([exemplo])[0, 1]:.3f}")

Valores positivos empurram para "não recomenda"; negativos, para "recomenda". A soma reproduz exatamente a previsão do modelo: a explicação é fiel. (Para modelos não lineares, o SHAP da aula 06 faz o mesmo papel.)

Para a tela, uma tabela com as palavras de maior efeito:

In [ ]:
def principais_palavras(texto, k=6):
    """As k palavras de maior efeito (em valor absoluto), prontas para um gr.Dataframe."""
    c = contribuicoes(texto)
    maiores = c.loc[c.abs().sort_values(ascending=False).index[:k]]
    return pd.DataFrame({
        "palavra": maiores.index,
        "efeito": maiores.round(2).values,
        "empurra para": np.where(maiores.values > 0, "não recomenda", "recomenda"),
    })


principais_palavras(exemplo)

### ✏️ Exercício 2 (★★): palavras realçadas no próprio texto

O componente `gr.HighlightedText` recebe uma lista de pares `(trecho, valor)` e colore cada trecho conforme o valor; trechos com valor `None` ficam sem cor.

Escreva `realcar(texto)`, que devolve essa lista para o texto recebido:

- separe o texto nos espaços (`texto.split()`), mantendo a ordem;
- para cada pedaço, procure a contribuição da palavra correspondente: o pedaço em minúsculas e sem a pontuação das pontas (`pedaco.lower().strip(string.punctuation)`);
- o valor do par é a contribuição (um `float`) se a palavra estiver em `contribuicoes(texto)`, e `None` caso contrário.

In [ ]:
# ✏️ Exercício 2
def realcar(texto):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
_texto = "Produto péssimo, não recomendo. Xyzabc!"
_pares = realcar(_texto)
assert isinstance(_pares, list) and [p for p, _ in _pares] == _texto.split(), "os trechos devem ser os pedaços do texto, na ordem"
_valores = dict(_pares)
assert _valores["Xyzabc!"] is None, "palavra desconhecida deve ter valor None"
assert isinstance(_valores["péssimo,"], float) and _valores["péssimo,"] > 0, "a pontuação das pontas deve ser ignorada ao procurar a palavra"
assert all(v is None or isinstance(v, float) for v in _valores.values())
print("Exercício 2 correto. Para ver o resultado na tela, execute a célula abaixo.")

In [ ]:
# Uma interface mínima para o exercício 2 (só funciona depois de resolvê-lo)
demo_realce = gr.Interface(
    fn=realcar,
    inputs=gr.Textbox(label="Avaliação", lines=3, value="O produto é bonito, mas chegou com defeito e o atendimento não resolveu."),
    outputs=gr.HighlightedText(label="Efeito de cada palavra", show_legend=True),
    flagging_mode="never",
)
# demo_realce.launch()      # retire o comentário para abrir

## 6. `Blocks`: uma aplicação de verdade

`gr.Interface` monta a tela sozinha, sempre no mesmo formato. Com `gr.Blocks`, você decide a disposição e liga cada evento à função que quiser.

| Conceito | Como se escreve |
|---|---|
| **Disposição** | blocos `with gr.Row():` e `with gr.Column():` aninhados; abas com `with gr.Tab("nome"):` |
| **Componentes** | `gr.Textbox`, `gr.Slider`, `gr.Label`, `gr.Dataframe`, `gr.Button`, `gr.Markdown`... |
| **Eventos** | `botao.click(fn=funcao, inputs=[...], outputs=[...])` |

A regra dos eventos: a função recebe **um argumento por componente de entrada**, na ordem da lista, e devolve **um valor por componente de saída**.

In [ ]:
def analisar_e_explicar(texto, limiar):
    """Função ligada ao botão: três saídas, uma para cada componente da direita."""
    probabilidades, decisao = analisar(texto, limiar)
    tabela = principais_palavras(texto) if probabilidades else pd.DataFrame(columns=["palavra", "efeito", "empurra para"])
    return probabilidades, decisao, tabela


SOBRE_O_MODELO = f"""
**Sobre este modelo.** Regressão logística sobre TF-IDF, treinada com avaliações de produtos de uma loja virtual brasileira,
coletadas de janeiro a maio de 2018 (B2W-Reviews01). AUC de {auc_teste:.2f} em avaliações não usadas no treino.
**Limites:** só entende português; olha palavras isoladas, sem ordem nem ironia; reflete o vocabulário e os produtos daquela loja, naquele período.
É uma demonstração didática, e não deve ser usada para decisões sobre pessoas.
"""

with gr.Blocks(title="Avaliações de clientes") as aplicacao:
    gr.Markdown("# Avaliações de clientes\nEstime se o autor de uma avaliação recomendaria o produto, e veja as palavras que mais pesaram.")
    with gr.Row():
        with gr.Column():                                                       # coluna da esquerda: entradas
            caixa = gr.Textbox(label="Avaliação do cliente", lines=5, placeholder="Cole ou digite uma avaliação")
            limiar = gr.Slider(0.05, 0.95, value=0.5, step=0.05, label="Limiar para encaminhar ao atendimento")
            botao = gr.Button("Analisar", variant="primary")
            gr.Examples(
                examples=["Chegou antes do prazo, produto excelente!", "Não recebi o produto e não consigo o reembolso.", "É bom, mas esperava mais pelo preço."],
                inputs=caixa,
            )
        with gr.Column():                                                       # coluna da direita: saídas
            saida_probabilidades = gr.Label(label="O cliente recomenda?")
            saida_decisao = gr.Textbox(label="Decisão")
            saida_palavras = gr.Dataframe(label="Palavras que mais pesaram")
    gr.Markdown(SOBRE_O_MODELO)

    botao.click(fn=analisar_e_explicar, inputs=[caixa, limiar], outputs=[saida_probabilidades, saida_decisao, saida_palavras])
    caixa.submit(fn=analisar_e_explicar, inputs=[caixa, limiar], outputs=[saida_probabilidades, saida_decisao, saida_palavras])

aplicacao.launch(quiet=True)

In [ ]:
aplicacao.close()

Repare no texto "Sobre este modelo", no pé da tela. Ele faz parte da aplicação tanto quanto o botão: diz com que dados o modelo foi treinado, quão bem ele funciona e onde não se deve confiar nele. Voltaremos a isso na seção 10.

## 7. Ouvir quem usa

Cada uso da demonstração é uma chance de descobrir onde o modelo erra. A forma mais simples de aproveitar é deixar a pessoa dizer se a resposta estava certa, e **guardar** essa informação: é o começo do conjunto de dados que vai melhorar a próxima versão.

O `gr.Interface` tem um botão pronto para isso (`flagging_mode="manual"`). Com `Blocks`, escrevemos a nossa própria função de registro, e decidimos exatamente o que é guardado.

### ✏️ Exercício 3 (★★★): registrar o retorno

**a)** Escreva `registrar_retorno(texto, probabilidade, opiniao, arquivo="retornos.csv")`:

- se o arquivo ainda não existe, cria-o com o cabeçalho `momento,texto,probabilidade,opiniao`;
- acrescenta uma linha com o momento atual (`datetime.now().isoformat(timespec="seconds")`), o texto, a probabilidade arredondada para 4 casas e a opinião;
- use o módulo `csv` (com `newline=""` e `encoding="utf-8"`), para que vírgulas e quebras de linha no texto não estraguem o arquivo;
- devolve a frase `"Obrigado! Retorno registrado."`.

**b)** Monte `demo_ex3`, um `gr.Blocks` com: uma caixa de texto, um botão "Analisar", um `gr.Label` com as probabilidades, dois botões ("Acertou" e "Errou") e uma caixa de texto de confirmação. Os dois botões de opinião devem chamar `registrar_retorno`.

Dicas: `gr.Number(visible=False)` serve para guardar a probabilidade entre um clique e outro; para passar um valor fixo (a opinião) a uma função, use `lambda`.

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
def registrar_retorno(texto, probabilidade, opiniao, arquivo="retornos.csv"):
    ...


with gr.Blocks() as demo_ex3:
    ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_arquivo = Path("retornos_teste.csv")
_arquivo.unlink(missing_ok=True)
assert registrar_retorno('Texto com vírgula, "aspas" e\nquebra de linha', 0.87654321, "errou", arquivo=_arquivo) == "Obrigado! Retorno registrado."
registrar_retorno("Segundo texto", 0.1, "acertou", arquivo=_arquivo)
_lido = pd.read_csv(_arquivo)
assert list(_lido.columns) == ["momento", "texto", "probabilidade", "opiniao"] and len(_lido) == 2, "o arquivo deve ter o cabeçalho uma única vez e uma linha por retorno"
assert _lido.loc[0, "texto"] == 'Texto com vírgula, "aspas" e\nquebra de linha', "o texto não foi preservado: use o módulo csv"
assert _lido.loc[0, "probabilidade"] == 0.8765 and _lido["opiniao"].tolist() == ["errou", "acertou"]
assert pd.to_datetime(_lido["momento"]).notna().all()
assert isinstance(demo_ex3, gr.Blocks) and sum(isinstance(b, gr.Button) for b in demo_ex3.blocks.values()) >= 3, "demo_ex3 deve ter os três botões"
_arquivo.unlink()
print("Exercício 3 correto. Para experimentar: demo_ex3.launch()")

## 8. Arquivos em lote

Quem gostou da demonstração logo pergunta: "posso mandar uma planilha?". A lógica é a mesma, com dois cuidados novos: **validar o arquivo** e **limitar o tamanho**.

### ✏️ Exercício 4 (★★): analisar um arquivo CSV

Escreva `analisar_arquivo(caminho, limiar=0.5)`:

- lê o CSV em `caminho` com o pandas;
- se não houver uma coluna chamada `texto`, levanta `gr.Error` com uma mensagem que diga qual coluna é esperada;
- considera no máximo as `MAX_LINHAS` primeiras linhas e trata textos ausentes como texto vazio;
- devolve um DataFrame com as colunas `texto`, `probabilidade` (de "não recomenda", arredondada para 3 casas) e `decisao` (`"encaminhar"` se a probabilidade for maior ou igual ao limiar, senão `"nenhuma ação"`).

Faça **uma única** chamada a `modelo.predict_proba` para o arquivo todo.

In [ ]:
MAX_LINHAS = 1000

In [ ]:
# ✏️ Exercício 4
def analisar_arquivo(caminho, limiar=0.5):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
pd.DataFrame({"texto": ["Péssimo, não recebi o produto.", "Excelente, recomendo!", None] * 500, "outra": 1}).to_csv("lote_teste.csv", index=False)
pd.DataFrame({"comentario": ["sem a coluna certa"]}).to_csv("lote_errado.csv", index=False)

_saida = analisar_arquivo("lote_teste.csv", limiar=0.5)
assert list(_saida.columns) == ["texto", "probabilidade", "decisao"] and len(_saida) == MAX_LINHAS, "confira as colunas e o limite de linhas"
assert _saida.loc[0, "decisao"] == "encaminhar" and _saida.loc[1, "decisao"] == "nenhuma ação"
assert _saida["texto"].isna().sum() == 0 and _saida["probabilidade"].between(0, 1).all()
assert (analisar_arquivo("lote_teste.csv", limiar=0.01)["decisao"] == "encaminhar").mean() > (_saida["decisao"] == "encaminhar").mean(), "o limiar deve alterar as decisões"
try:
    analisar_arquivo("lote_errado.csv")
    raise AssertionError("um arquivo sem a coluna 'texto' deveria gerar gr.Error")
except gr.Error as _erro:
    assert "texto" in str(_erro), "a mensagem de erro deve dizer qual coluna é esperada"
for _nome in ["lote_teste.csv", "lote_errado.csv"]:
    Path(_nome).unlink()
print("Exercício 4 correto. Na tela, a função se liga a um gr.File de entrada e a um gr.Dataframe de saída:")
print('    gr.Interface(fn=analisar_arquivo, inputs=[gr.File(file_types=[".csv"]), gr.Slider(0.05, 0.95, value=0.5)], outputs=gr.Dataframe())')

## 9. Do notebook ao `app.py`

Uma aplicação publicada é um arquivo Python comum, acompanhado do artefato e da lista de dependências. A estrutura mínima:

```
minha-demo/
├── app.py                     # a aplicação
├── modelo_avaliacoes.joblib   # o artefato
└── requirements.txt           # as bibliotecas, com versões
```

A célula abaixo escreve os três arquivos. O `app.py` contém só o que a aplicação precisa: carregar o modelo, as funções e a tela. Nada de treino.

In [ ]:
PASTA_APP = Path("minha-demo")
PASTA_APP.mkdir(exist_ok=True)

CODIGO_APP = '''"""Demonstração: o autor de uma avaliação recomendaria o produto?"""
import gradio as gr
import joblib
import numpy as np
import pandas as pd

modelo = joblib.load("modelo_avaliacoes.joblib")
vetorizador, classificador = modelo.named_steps["tfidf"], modelo.named_steps["clf"]
MAX_CARACTERES = 5000


def principais_palavras(texto, k=6):
    linha = vetorizador.transform([texto])
    valores = linha.multiply(classificador.coef_).toarray()[0]
    presentes = linha.nonzero()[1]
    c = pd.Series(valores[presentes], index=vetorizador.get_feature_names_out()[presentes])
    maiores = c.loc[c.abs().sort_values(ascending=False).index[:k]]
    return pd.DataFrame({"palavra": maiores.index, "efeito": maiores.round(2).values,
                         "empurra para": np.where(maiores.values > 0, "não recomenda", "recomenda")})


def analisar(texto, limiar=0.5):
    texto = (texto or "").strip()[:MAX_CARACTERES]
    if not texto:
        raise gr.Error("Digite uma avaliação antes de analisar.")
    if vetorizador.transform([texto]).nnz == 0:
        return {}, "Não foi possível analisar: nenhuma palavra do texto é conhecida pelo modelo.", None
    p = float(modelo.predict_proba([texto])[0, 1])
    decisao = "Encaminhar para o atendimento" if p >= limiar else "Nenhuma ação necessária"
    return {"não recomenda": p, "recomenda": 1 - p}, f"{decisao} (probabilidade {p:.0%}, limiar {limiar:.0%})", principais_palavras(texto)


with gr.Blocks(title="Avaliações de clientes") as demo:
    gr.Markdown("# Avaliações de clientes")
    with gr.Row():
        with gr.Column():
            caixa = gr.Textbox(label="Avaliação do cliente", lines=5)
            limiar = gr.Slider(0.05, 0.95, value=0.5, step=0.05, label="Limiar para encaminhar ao atendimento")
            botao = gr.Button("Analisar", variant="primary")
        with gr.Column():
            saidas = [gr.Label(label="O cliente recomenda?"), gr.Textbox(label="Decisão"), gr.Dataframe(label="Palavras que mais pesaram")]
    gr.Markdown("Modelo didático, treinado com avaliações de 2018 de uma loja virtual brasileira. Só entende português.")
    botao.click(fn=analisar, inputs=[caixa, limiar], outputs=saidas)

if __name__ == "__main__":
    demo.launch()
'''

(PASTA_APP / "app.py").write_text(CODIGO_APP, encoding="utf-8")
(PASTA_APP / "requirements.txt").write_text(
    f"gradio=={gr.__version__}\nscikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\npandas=={pd.__version__}\nnumpy=={np.__version__}\n",
    encoding="utf-8",
)
joblib.dump(modelo, PASTA_APP / "modelo_avaliacoes.joblib")

print((PASTA_APP / "requirements.txt").read_text())
print(sorted(p.name for p in PASTA_APP.iterdir()))

O `requirements.txt` registra as versões **exatas** deste ambiente. É o que garante que o `.joblib` seja lido pela mesma versão do scikit-learn que o gravou.

Antes de publicar, um teste: executar o `app.py` como um programa separado, do zero, só com os três arquivos. Se ele importa e responde, a aplicação não depende de nada que ficou esquecido no notebook.

In [ ]:
teste = '''
import app
probabilidades, decisao, palavras = app.analisar("Não recebi o produto, péssimo.", 0.5)
print(decisao, "|", len(palavras), "palavras na tabela |", type(app.demo).__name__)
'''
resultado = subprocess.run([sys.executable, "-c", teste], cwd=PASTA_APP, capture_output=True, text=True)
print(resultado.stdout.strip() or resultado.stderr[-800:])
assert resultado.returncode == 0, "o app.py não funciona sozinho"

Com esses três arquivos, publicar é copiar a pasta para um serviço que execute `python app.py`. Os slides de entrega (deck D) mostram o passo a passo no *Hugging Face Spaces* e a alternativa com Streamlit. Atenção às condições de gratuidade, que mudaram em 2026: hoje uma conta gratuita do Hugging Face só pode criar até duas aplicações Gradio, no hardware *ZeroGPU*, e precisa ter e-mail verificado e mais de 30 dias; confira a regra vigente antes de prometer um link a alguém.

## 10. O que uma demonstração deve avisar, e o que não deve guardar

Uma demonstração é, para muita gente, o primeiro contato com o modelo. O que ela mostra e o que ela omite formam a ideia que as pessoas farão dele.

| Pergunta | O que fazer na tela |
|---|---|
| Com que dados o modelo foi treinado? | dizer origem, período e idioma |
| Quão bem ele funciona? | uma métrica, medida em dados não usados no treino, com o que ela significa |
| Onde ele falha? | listar limites conhecidos; não responder quando a entrada está fora do previsto |
| Para que não deve ser usado? | dizer explicitamente (decisões sobre pessoas, por exemplo) |
| A resposta é certa? | mostrar probabilidade e explicação, e não só o rótulo |
| O que acontece com o que eu digito? | avisar se o texto é guardado, e para quê |

Sobre **guardar dados**: o texto digitado pode conter nomes, telefones, endereços. Guarde o mínimo necessário para o objetivo declarado, avise na tela, e não registre nada "para o caso de precisar". A LGPD se aplica a demonstrações como a qualquer outro sistema.

### ✏️ Exercício 5 (★★): revisão crítica

Um colega publicou uma demonstração com esta descrição: *"IA que detecta clientes insatisfeitos com 96% de precisão. Cole a avaliação e descubra!"*. A tela mostra apenas o rótulo "INSATISFEITO" ou "SATISFEITO", e todas as entradas são gravadas "para melhorar o modelo".

Aponte **quatro problemas** e proponha a correção de cada um.

✏️ **Sua resposta:**

_Escreva aqui._

## Para levar

### Erros comuns

| Erro | Consequência | Como evitar |
|---|---|---|
| Treinar o modelo dentro da aplicação | inicialização lenta; resultado muda a cada reinício | aplicação só carrega o artefato |
| Não fixar as versões das bibliotecas | o `.joblib` não carrega, ou carrega com resultado diferente | `requirements.txt` com versões exatas |
| Função de previsão testada só pela tela | erros aparecem diante do usuário | testar a função como Python comum |
| Aceitar qualquer entrada | respostas sem sentido com aparência de certeza | validar, limitar o tamanho, saber dizer "não sei" |
| Mostrar só o rótulo | confiança cega ou desconfiança total | probabilidade, limiar e explicação |
| Número de argumentos da função diferente do número de entradas | erro ao clicar no botão | um argumento por componente de entrada, um retorno por saída |
| Compartilhar o link público sem pensar | sua máquina e seus dados ficam acessíveis | `auth=`, ou não compartilhar |
| Gravar tudo o que o usuário digita | risco de privacidade | mínimo necessário, com aviso |
| Abrir um `.joblib` recebido de terceiros | execução de código arbitrário | só carregar artefatos de origem confiável |

### Resumo

- A aplicação **carrega um artefato**; treino e entrega ficam em arquivos separados.
- O centro de tudo é uma **função Python testada**: valida a entrada, prevê, decide e sabe recusar.
- `gr.Interface` resolve o caso simples; `gr.Blocks` dá controle sobre disposição e eventos.
- **Explicar** a previsão é parte da interface. Em modelos lineares a explicação é exata; nos demais, SHAP.
- O retorno dos usuários é dado valioso e é dado de terceiros: registre com critério.
- Publicar é levar `app.py`, artefato e `requirements.txt` com versões exatas.
- A tela deve dizer de onde o modelo veio, quão bem funciona e onde não confiar nele.

### O percurso do curso

| Aulas | O que você construiu |
|---|---|
| 00 a 04 | as ferramentas: Python, NumPy, pandas, visualização, Polars e DuckDB |
| 05 e 06 | modelos para tabelas, com validação honesta e interpretação |
| 07 e 08 | redes neurais do zero: do tensor ao treino completo |
| 09 e 10 | modelos pré-treinados e LLMs como componentes |
| 11 | o modelo nas mãos de quem vai usar |

### Para casa

A lista de exercícios está em `11_casa.ipynb`: uma demonstração para a rede de imagens da aula 08.